# Geosteering AI v2.40 — Treinamento com Mixed Precision + XLA

**Template**: `train_v240_mp16.ipynb` | **Sprint**: v2.40 (I2.2 MCP colab-bridge)

Treina um modelo ResNet 18 do Geosteering AI com:
- `mixed_float16` (compute fp16, master weights fp32) via `setup_mixed_precision_policy(config)`
- XLA JIT compilation
- `tf.data` otimizado (AUTOTUNE + cache + prefetch + custom shuffle buffer)

**Pré-requisito local**: API REST `geosteering-api` rodando + ngrok ativo (para smoke test final).

**Saída**: `model.keras` + `scalers.joblib` + `history.json` no Drive.

## Cell 1 — Configuração

Edite as variáveis abaixo antes de executar.

In [ ]:
# Configuração do notebook (parametrizável)
GIT_TAG = "v2.40"
DATASET_DRIVE_PATH = "/content/drive/MyDrive/Geosteering_AI/datasets/baseline.h5"
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/Geosteering_AI/v2.40/train_runs/"

# Configuração de treinamento
MODEL_TYPE = "ResNet_18"
EPOCHS = 50
BATCH_SIZE = 64  # T4 suporta 64-128 com mp16
USE_MIXED_PRECISION = True
USE_XLA = True
SEED = 42

# Smoke test final (opcional): URL ngrok da API REST local
COLAB_API_BASE_URL = ""  # ex.: "https://abc123.ngrok.app"

print(f"✓ Configuração: model={MODEL_TYPE}, epochs={EPOCHS}, batch={BATCH_SIZE}")
print(f"  mixed_precision={USE_MIXED_PRECISION}, XLA={USE_XLA}")

## Cell 2 — Setup do Ambiente

In [ ]:
# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

# Clonar repo na tag v2.40
!git clone --depth 1 --branch {GIT_TAG} https://github.com/daniel-guitarplayer-8/geosteering-ai.git
%cd geosteering-ai

# Instalar com extras [all] (TF + JAX + viz + train)
!pip install -q -e ".[all]"

# Validar GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
assert len(gpus) > 0, "Nenhuma GPU detectada — verifique Runtime → Change runtime type → GPU"
print(f"✓ GPU TF: {gpus[0].name}")
print(f"✓ TF version: {tf.__version__}")
# Sprint v2.40.1 (Security #6 + hardening F1): pin commit hash imutável
# Tag pode ser movida; hash é imutável → rastreabilidade total do run.
# try/except graceful: se repo foi baixado como ZIP (sem .git/) ou
# instalado via pip git+... (sem clone), usa fallback baseado em tag.
import os, subprocess
_REPO_DIR = "/content/geosteering-ai"
try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=_REPO_DIR if os.path.isdir(_REPO_DIR + "/.git") else ".",
    ).decode().strip()
    print(f"✓ Pinned commit: {GIT_COMMIT[:12]}")
except (subprocess.CalledProcessError, FileNotFoundError) as _e:
    GIT_COMMIT = f"unknown-from-tag-{GIT_TAG}"
    print(f"⚠ git rev-parse falhou ({_e}); fallback: {GIT_COMMIT}")


## Cell 3 — Configurar Mixed Precision Policy (ANTES de build_model)

**CRÍTICO**: D5 da Sprint v2.40 — a ordem importa.
- ❌ Errado: `build_model()` → `set_global_policy(mixed_float16)` → camadas ficam fp32
- ✅ Correto: `set_global_policy(mixed_float16)` → `build_model()` → camadas fp16

In [ ]:
from geosteering_ai.config import PipelineConfig
from geosteering_ai.training.loop import setup_mixed_precision_policy

# Construir config com flags de v2.40
config = PipelineConfig(
    model_type=MODEL_TYPE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    use_mixed_precision=USE_MIXED_PRECISION,
    use_xla=USE_XLA,
    # Flags tf.data novas em v2.40 (D6 do plano)
    tf_shuffle_buffer_size=10000,
    tf_num_parallel_calls=-1,   # -1 = AUTOTUNE
    tf_prefetch_buffer_size=-1, # -1 = AUTOTUNE
    tf_cache_eval=True,
    seed=SEED,
)

# CRÍTICO: ANTES de qualquer build_model
setup_mixed_precision_policy(config)
print(f"✓ Policy ativa: {tf.keras.mixed_precision.global_policy().name}")

## Cell 4 — Carregar Dataset

In [ ]:
from geosteering_ai.data.pipeline import DataPipeline

pipeline = DataPipeline(config)
data = pipeline.prepare(DATASET_DRIVE_PATH)

print(f"✓ Train: {data['train_raw'].shape}")
print(f"✓ Val:   {data['val'][0].shape}")
print(f"✓ Test:  {data['test'][0].shape}")

## Cell 5 — Construir Modelo com Policy Correta

In [ ]:
from geosteering_ai.models.registry import build_model_with_mp_policy

# Helper de v2.40 (D5): garante setup_mixed_precision_policy() ANTES de build
model = build_model_with_mp_policy(config)

# Verificar compute_dtype das camadas
if USE_MIXED_PRECISION:
    assert model.compute_dtype == "float16", (
        f"compute_dtype esperado float16, atual={model.compute_dtype}"
    )
print(f"✓ Modelo construído: {model.name}")
print(f"✓ compute_dtype: {model.compute_dtype}")
print(f"✓ variable_dtype: {model.variable_dtype}")
model.summary()

## Cell 6 — Treinar

In [ ]:
from geosteering_ai.training.loop import TrainingLoop
import time

loop = TrainingLoop(config=config, model=model)
t0 = time.perf_counter()
history = loop.run(data)
elapsed = time.perf_counter() - t0

print(f"\n✓ Treinamento concluído em {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"  Final loss: {history.history['loss'][-1]:.6f}")
print(f"  Final val_loss: {history.history['val_loss'][-1]:.6f}")

## Cell 7 — Salvar Artefatos no Drive

In [ ]:
import os
import json
import datetime
from geosteering_ai.inference.pipeline import InferencePipeline

# Diretório com timestamp para esta run
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out_dir = os.path.join(OUTPUT_DRIVE_DIR, f"run_{ts}")
os.makedirs(out_dir, exist_ok=True)

# Salvar pipeline completo (model.keras + scalers.joblib + config.yaml)
inference = InferencePipeline(model=model, config=config, scaler=pipeline.scaler)
inference.save(out_dir)

# Salvar history como JSON
history_json = {k: [float(x) for x in v] for k, v in history.history.items()}
with open(os.path.join(out_dir, "history.json"), "w") as f:
    json.dump({
        "template": "train_v240_mp16",
        "git_tag": GIT_TAG,
        "git_commit": GIT_COMMIT,  # v2.40.1 Security #6
        "runtime": "colab_pro_plus",
        "timestamp_utc": ts,
        "config": {
            "model_type": MODEL_TYPE,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "use_mixed_precision": USE_MIXED_PRECISION,
            "use_xla": USE_XLA,
        },
        "history": history_json,
        "duration_sec": elapsed,
        "exit_code": 0,
    }, f, indent=2)

print(f"✓ Artefatos salvos em {out_dir}")

## Cell 8 — Smoke Test via API REST (Opcional)

Se `COLAB_API_BASE_URL` está setado (ngrok rodando local), valida o pipeline
via `POST /predict` para confirmar que tudo está consistente.

In [ ]:
if COLAB_API_BASE_URL:
    import requests
    import numpy as np
    
    # Health check (hardening v2.40 review security #3: sem body p/ evitar vazamento)
    r = requests.get(f"{COLAB_API_BASE_URL}/health", timeout=10)
    print(f"GET /health → {r.status_code}")
    
    # Pegar 1 amostra real do val_raw
    sample = data["val_raw"][:1].tolist()
    r = requests.post(
        f"{COLAB_API_BASE_URL}/predict",
        json={"raw_data": sample},
        timeout=30,
    )
    print(f"POST /predict → {r.status_code}")
    if r.status_code == 200:
        body = r.json()
        print(f"  shape: {body['shape']}")
        print(f"  latency_ms: {body['latency_ms']:.2f}")
        print("✓ Smoke test API REST PASS")
else:
    print("COLAB_API_BASE_URL vazio — pulando smoke test (configure ngrok para ativar)")


## Cell 9 — Resumo

In [ ]:
print("=" * 60)
print("SPRINT v2.40 — TREINAMENTO mp16+XLA CONCLUÍDO")
print("=" * 60)
print(f"Modelo:           {MODEL_TYPE}")
print(f"compute_dtype:    {model.compute_dtype}")
print(f"variable_dtype:   {model.variable_dtype}")
print(f"Épocas:           {EPOCHS}")
print(f"Batch:            {BATCH_SIZE}")
print(f"Duração:          {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Final val_loss:   {history.history['val_loss'][-1]:.6f}")
print(f"Artefatos:        {out_dir}")
print("=" * 60)